In [1]:
#Install dependencies
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes triton
!pip install mlflow==2.19.0 rouge-score bert-score

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-m6vrpx22/unsloth_120e97ffd0a84951ad0450cd5f0dbb02
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-m6vrpx22/unsloth_120e97ffd0a84951ad0450cd5f0dbb02
  Resolved https://github.com/unslothai/unsloth.git to commit b09aa82a3ac9ac9794c497e4b8f747f77e52b162
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached pyarrow-24.0.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (3.0 kB)
Using cached pyarrow-24.0.0-cp312-cp312-manylinux_2_28_x86_64.whl (48.9 MB)
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following depe

In [2]:
# Load data
from google.colab import drive
drive.mount('/content/drive')

import json
save_dir = "/content/drive/MyDrive/review-summarizer"

with open(f"{save_dir}/test_data.json") as f:
    test_data = json.load(f)

print(f"Test examples: {len(test_data)}")
print(f"\nSample:")
print(f"  Input: {test_data[0]['input'][:150]}...")
print(f"  Output: {test_data[0]['output'][:150]}...")

Mounted at /content/drive
Test examples: 187

Sample:
  Input: Review: I was asked to watch this movie over the summer for my AP US History class. Basically, I was watching it for the historical aspect. However, h...
  Output: **Pros:**
- None mentioned

**Cons:**
- Lacked historical accuracy
- Focused on romance over facts
- Ignored effects of the voyage
- Unrealistic drama...


In [3]:
# Load fine-tuned model (base + winning adapters)
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    max_seq_length=2048, dtype=None, load_in_4bit=True,
)

# Apply LoRA structure
model = FastLanguageModel.get_peft_model(
    model, r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16, lora_dropout=0, bias="none",
    use_gradient_checkpointing="unsloth", random_state=42,
)

# Load winning adapter weights
adapter_path = f"{save_dir}/adapters_run4-2ep-lr1.5e4-r16"
model.load_adapter(adapter_path, adapter_name="default")
print(f"Fine-tuned model loaded from: {adapter_path}")
print(f"GPU memory: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.8: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/4.14G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/157 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/446 [00:00<?, ?B/s]

Unsloth 2026.4.8 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


Fine-tuned model loaded from: /content/drive/MyDrive/review-summarizer/adapters_run4-2ep-lr1.5e4-r16
GPU memory: 4.4 GB


In [4]:
# Generate predictions — fine-tuned model
FastLanguageModel.for_inference(model)

def generate_summary(model, tokenizer, instruction, input_text):
    prompt = f"""<s>[INST] {instruction}

{input_text} [/INST]
"""
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(
        **inputs, max_new_tokens=256,
        temperature=0.7, top_p=0.9, repetition_penalty=1.1,
    )
    response = tokenizer.decode(
        outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True
    )
    return response.strip()

# Generate predictions for ALL test examples
finetuned_predictions = []
for i, example in enumerate(test_data):
    if i % 25 == 0:
        print(f"Fine-tuned model: {i}/{len(test_data)}...")
    pred = generate_summary(model, tokenizer, example['instruction'], example['input'])
    finetuned_predictions.append(pred)

print(f"\nDone! {len(finetuned_predictions)} predictions generated")
print(f"\nSample prediction:")
print(finetuned_predictions[0])

Fine-tuned model: 0/187...


Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12

Fine-tuned model: 25/187...


Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

Fine-tuned model: 50/187...


...

Fine-tuned model: 150/187...


Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

Fine-tuned model: 175/187...


Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene


Done! 187 predictions generated

Sample prediction:
**Pros:**
- No notable strengths mentioned

**Cons:**
- Lacked historical accuracy
- Focused more on romance than history
- Inaccurate portrayal of disease
- Unrealistic character conflicts

**Verdict:** This movie fails to deliver accurate historical content.


In [5]:
# Generate predictions — BASE model (no fine-tuning)
# This is the comparison baseline — same model WITHOUT our training

# Need to reload model fresh without adapters
del model
torch.cuda.empty_cache()

base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    max_seq_length=2048, dtype=None, load_in_4bit=True,
)
FastLanguageModel.for_inference(base_model)

base_predictions = []
for i, example in enumerate(test_data):
    if i % 25 == 0:
        print(f"Base model: {i}/{len(test_data)}...")
    pred = generate_summary(base_model, base_tokenizer, example['instruction'], example['input'])
    base_predictions.append(pred)

print(f"\nDone! {len(base_predictions)} base model predictions generated")
print(f"\nSample base prediction:")
print(base_predictions[0])

# Free base model memory
del base_model, base_tokenizer
torch.cuda.empty_cache()

==((====))==  Unsloth 2026.4.8: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Base model: 0/187...


Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

...

Base model: 150/187...


Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

Base model: 175/187...


Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene


Done! 187 base model predictions generated

Sample base prediction:
**Summary:**

*Title:* Adaptation of Columbus Voyage
*Rating:* 1/5

**Pros:**
- The portrayal of romantic relationships between Columbus and his wife.
- Depiction of conflicts among men and their personal struggles.

**Cons:**
1. Lack of historical accuracy: The reviewer found the movie to be more focused on romance and drama rather than providing an accurate account of the historical events.
2. Omission of significant historical aspects: The reviewer criticized the absence of the impact of the voyage, particularly the spread of diseases, which is a crucial part of the Columbian Exchange.
3. Inaccurate representation of interactions with indigenous peoples: The depiction of European men fighting over Native American women was seen as unrealistic and insensitive.
4. Exaggerated plotlines and unnecessary violence: The reviewer felt that the fights among crew members were unrealistic and added no value to the story.
5. M

In [6]:
# Save predictions to Drive
predictions_data = {
    "finetuned": finetuned_predictions,
    "base": base_predictions,
    "references": [ex['output'] for ex in test_data],
    "inputs": [ex['input'] for ex in test_data],
}

with open(f"{save_dir}/predictions.json", "w") as f:
    json.dump(predictions_data, f)
print("Predictions saved to Drive!")

Predictions saved to Drive!


In [7]:
# Compute ROUGE-L
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
references = [ex['output'] for ex in test_data]

# Fine-tuned model ROUGE-L
ft_rouge_scores = []
for pred, ref in zip(finetuned_predictions, references):
    score = scorer.score(ref, pred)
    ft_rouge_scores.append(score['rougeL'].fmeasure)

avg_ft_rouge = sum(ft_rouge_scores) / len(ft_rouge_scores)

# Base model ROUGE-L
base_rouge_scores = []
for pred, ref in zip(base_predictions, references):
    score = scorer.score(ref, pred)
    base_rouge_scores.append(score['rougeL'].fmeasure)

avg_base_rouge = sum(base_rouge_scores) / len(base_rouge_scores)

# Improvement
improvement = ((avg_ft_rouge - avg_base_rouge) / avg_base_rouge) * 100

print(f"ROUGE-L Results:")
print(f"  Base model:      {avg_base_rouge:.4f}")
print(f"  Fine-tuned:      {avg_ft_rouge:.4f}")
print(f"  Improvement:     {improvement:.1f}%")

ROUGE-L Results:
  Base model:      0.2115
  Fine-tuned:      0.5711
  Improvement:     170.1%


In [9]:
# Compute BERTScore (using lighter model)
from bert_score import score as bert_score_fn

# Fine-tuned model BERTScore
print("Computing BERTScore for fine-tuned model...")
ft_P, ft_R, ft_F1 = bert_score_fn(
    finetuned_predictions, references,
    lang="en",
    model_type="roberta-large",  # Lighter, works on T4
    verbose=True,
    batch_size=8,
)
avg_ft_bertscore = ft_F1.mean().item()
print(f"Fine-tuned BERTScore F1: {avg_ft_bertscore:.4f}")

# Base model BERTScore
print("\nComputing BERTScore for base model...")
base_P, base_R, base_F1 = bert_score_fn(
    base_predictions, references,
    lang="en",
    model_type="roberta-large",
    verbose=True,
    batch_size=8,
)
avg_base_bertscore = base_F1.mean().item()

bert_improvement = ((avg_ft_bertscore - avg_base_bertscore) / avg_base_bertscore) * 100

print(f"\nBERTScore Results:")
print(f"  Base model:      {avg_base_bertscore:.4f}")
print(f"  Fine-tuned:      {avg_ft_bertscore:.4f}")
print(f"  Improvement:     {bert_improvement:.1f}%")

Computing BERTScore for fine-tuned model...


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

calculating scores...
computing bert embedding.


  0%|          | 0/47 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/24 [00:00<?, ?it/s]

done in 9.24 seconds, 20.25 sentences/sec

BERTScore Results:
  Base model:      0.8655
  Fine-tuned:      0.9540
  Improvement:     10.2%


In [ ]:
# LLM-as-Judge (uses OpenAI API)
import openai

client = openai.OpenAI(api_key="")

JUDGE_PROMPT = """You are evaluating a product review summarizer.

ORIGINAL REVIEW: {review}

REFERENCE SUMMARY: {reference}

MODEL OUTPUT: {prediction}

Score the model output on these criteria (1-5 each):
1. STRUCTURE: Does it have clear Pros, Cons, and Verdict sections?
2. ACCURACY: Are the pros/cons actually mentioned in the review?
3. COMPLETENESS: Does it capture the key points?
4. CONCISENESS: Is it appropriately brief without losing meaning?

Respond in JSON only:
{{"structure": N, "accuracy": N, "completeness": N, "conciseness": N, "overall": N}}"""

import random
random.seed(42)
sample_indices = random.sample(range(len(test_data)), min(50, len(test_data)))

# Score fine-tuned model
ft_judge_scores = []
for i, idx in enumerate(sample_indices):
    if i % 10 == 0:
        print(f"Judging fine-tuned: {i}/{len(sample_indices)}...")
    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": JUDGE_PROMPT.format(
                review=test_data[idx]['input'],
                reference=references[idx],
                prediction=finetuned_predictions[idx]
            )}],
            temperature=0, max_tokens=100,
            response_format={"type": "json_object"}
        )
        score = json.loads(response.choices[0].message.content)
        ft_judge_scores.append(score)
    except Exception as e:
        print(f"  Error: {e}")

# Score base model
base_judge_scores = []
for i, idx in enumerate(sample_indices):
    if i % 10 == 0:
        print(f"Judging base: {i}/{len(sample_indices)}...")
    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": JUDGE_PROMPT.format(
                review=test_data[idx]['input'],
                reference=references[idx],
                prediction=base_predictions[idx]
            )}],
            temperature=0, max_tokens=100,
            response_format={"type": "json_object"}
        )
        score = json.loads(response.choices[0].message.content)
        base_judge_scores.append(score)
    except Exception as e:
        print(f"  Error: {e}")

# Calculate averages
def avg_scores(scores_list):
    result = {}
    for key in ['structure', 'accuracy', 'completeness', 'conciseness', 'overall']:
        values = [s[key] for s in scores_list if key in s]
        result[key] = sum(values) / len(values) if values else 0
    return result

ft_avg = avg_scores(ft_judge_scores)
base_avg = avg_scores(base_judge_scores)

print(f"\nLLM-as-Judge Results (1-5 scale):")
print(f"{'Metric':<16} {'Base':>8} {'Fine-tuned':>12} {'Diff':>8}")
print(f"{'-'*44}")
for key in ['structure', 'accuracy', 'completeness', 'conciseness', 'overall']:
    print(f"{key:<16} {base_avg[key]:>8.2f} {ft_avg[key]:>12.2f} {ft_avg[key]-base_avg[key]:>+8.2f}")

Judging fine-tuned: 0/50...
Judging fine-tuned: 10/50...
Judging fine-tuned: 20/50...
Judging fine-tuned: 30/50...
Judging fine-tuned: 40/50...
Judging base: 0/50...
Judging base: 10/50...
Judging base: 20/50...
Judging base: 30/50...
Judging base: 40/50...

LLM-as-Judge Results (1-5 scale):
Metric               Base   Fine-tuned     Diff
--------------------------------------------
structure            4.98         5.00    +0.02
accuracy             4.54         4.40    -0.14
completeness         4.80         4.14    -0.66
conciseness          3.96         4.76    +0.80
overall              4.62         4.36    -0.26


In [11]:
# Print complete evaluation summary
print(f"""
{'='*60}
EVALUATION RESULTS SUMMARY
{'='*60}

ROUGE-L F1:
  Base model:       {avg_base_rouge:.4f}
  Fine-tuned:       {avg_ft_rouge:.4f}
  Improvement:      {improvement:.1f}%

BERTScore F1:
  Base model:       {avg_base_bertscore:.4f}
  Fine-tuned:       {avg_ft_bertscore:.4f}
  Improvement:      {bert_improvement:.1f}%

LLM-as-Judge (1-5):
  Base overall:     {base_avg['overall']:.2f}
  Fine-tuned:       {ft_avg['overall']:.2f}

Test set size:      {len(test_data)}
Judge sample size:  {len(sample_indices)}
{'='*60}
""")


EVALUATION RESULTS SUMMARY

ROUGE-L F1:
  Base model:       0.2115
  Fine-tuned:       0.5711
  Improvement:      170.1%

BERTScore F1:
  Base model:       0.8655
  Fine-tuned:       0.9540
  Improvement:      10.2%

LLM-as-Judge (1-5):
  Base overall:     4.62
  Fine-tuned:       4.36

Test set size:      187
Judge sample size:  50



In [12]:
# Save all metrics to Drive
eval_results = {
    "rouge_l": {
        "base": avg_base_rouge,
        "finetuned": avg_ft_rouge,
        "improvement_pct": improvement,
    },
    "bertscore": {
        "base": avg_base_bertscore,
        "finetuned": avg_ft_bertscore,
        "improvement_pct": bert_improvement,
    },
    "llm_judge": {
        "base": base_avg,
        "finetuned": ft_avg,
    },
    "test_set_size": len(test_data),
    "judge_sample_size": len(sample_indices),
}

with open(f"{save_dir}/eval_results.json", "w") as f:
    json.dump(eval_results, f, indent=2)
print("Evaluation results saved to Drive!")

Evaluation results saved to Drive!


In [13]:
# Compare actual outputs side by side
import random
random.seed(42)

samples = random.sample(range(len(test_data)), 5)

for idx in samples:
    print(f"\n{'='*60}")
    print(f"REVIEW: {test_data[idx]['input'][:200]}...")
    print(f"\n--- BASE MODEL ---")
    print(base_predictions[idx][:300])
    print(f"\n--- FINE-TUNED ---")
    print(finetuned_predictions[idx][:300])
    print(f"\n--- REFERENCE ---")
    print(test_data[idx]['output'][:300])


REVIEW: Review: In Percy's Chocolate Crunch, the first episode really shows how an engine can get dirty. In Thomas, Percy and the Squeak, even if an engine is dirty, they can still do big jobs. In Gordon Take...

--- BASE MODEL ---
Title: Review of Thomas & Friends - Various Episodes

Pros:
1. Engaging Storytelling: The episodes provide interesting and engaging stories that showcase various aspects of train life, such as the importance of cleanliness, adaptability, and taking on new challenges.
2. Humor and Morals: The reviews

--- FINE-TUNED ---
**Pros:**
- Funny scenes throughout the episodes
- Good moral lessons for all ages
- One of the best Thomas videos yet

**Cons:**
- No significant drawbacks mentioned

**Verdict:** This video is highly recommended for fans of Thomas.

--- REFERENCE ---
**Pros:**
- Funny scenes
- Good morals
- Suitable for all ages
- One of the best Thomas videos

**Cons:**
- No significant drawbacks mentioned

**Verdict:** This video is highly recommended for

In [14]:
# Log evaluation metrics to MLflow
import mlflow, os

os.system(f"cp {save_dir}/mlflow_fresh.db /tmp/mlflow_fresh.db")
mlflow.set_tracking_uri("sqlite:////tmp/mlflow_fresh.db")
mlflow.set_experiment("product-review-summarizer")

with mlflow.start_run(run_name="evaluation-final"):
    mlflow.log_metrics({
        "rouge_l_base": avg_base_rouge,
        "rouge_l_finetuned": avg_ft_rouge,
        "rouge_l_improvement_pct": improvement,
        "bertscore_base": avg_base_bertscore,
        "bertscore_finetuned": avg_ft_bertscore,
        "bertscore_improvement_pct": bert_improvement,
        "judge_base_overall": base_avg['overall'],
        "judge_finetuned_overall": ft_avg['overall'],
        "judge_finetuned_structure": ft_avg['structure'],
        "judge_finetuned_conciseness": ft_avg['conciseness'],
        "judge_finetuned_accuracy": ft_avg['accuracy'],
        "judge_finetuned_completeness": ft_avg['completeness'],
        "test_set_size": len(test_data),
        "judge_sample_size": len(sample_indices),
    })
    mlflow.log_params({
        "eval_model": "run4-retrain-2ep-lr1.5e4-r16",
        "bertscore_model": "roberta-large",
        "judge_model": "gpt-4o-mini",
    })
    mlflow.set_tag("type", "evaluation")
print("Evaluation metrics logged to MLflow!")

# Save MLflow DB to Drive
os.system(f"cp /tmp/mlflow_fresh.db {save_dir}/mlflow_fresh.db")
print("MLflow DB saved!")

Evaluation metrics logged to MLflow!
MLflow DB saved!


In [15]:
# Save evaluation results to Drive
eval_results = {
    "rouge_l": {
        "base": avg_base_rouge,
        "finetuned": avg_ft_rouge,
        "improvement_pct": improvement,
    },
    "bertscore": {
        "base": avg_base_bertscore,
        "finetuned": avg_ft_bertscore,
        "improvement_pct": bert_improvement,
    },
    "llm_judge": {
        "base": base_avg,
        "finetuned": ft_avg,
    },
    "test_set_size": len(test_data),
    "judge_sample_size": len(sample_indices),
}

with open(f"{save_dir}/eval_results.json", "w") as f:
    json.dump(eval_results, f, indent=2)
print("Evaluation results saved!")

# Save sample comparisons for GitHub README
samples = []
for idx in random.sample(range(len(test_data)), 5):
    samples.append({
        "input": test_data[idx]['input'],
        "reference": test_data[idx]['output'],
        "finetuned": finetuned_predictions[idx],
        "base": base_predictions[idx],
    })

with open(f"{save_dir}/sample_comparisons.json", "w") as f:
    json.dump(samples, f, indent=2)
print("Sample comparisons saved!")

Evaluation results saved!
Sample comparisons saved!
